In [1]:
import requests
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt
import os

# Main

In [2]:
API_KEY = os.environ['KEY']
SeriesIDs = ['UNRATE', 'PAYEMS', 'ICSA', 'INDPRO', 'JTSJOL']

## Get data for db table `FREDSeries`
- SeriesID
- Title 
- DefaultFrequency
- Units
- ObservationStart
- FRED_LastUpdateCD
- FRED_LastUpdateUTC (must be converted from CD/CDT to UTC)
- CreationDate `TEXT NOT NULL DEFAULT (datetime('now'))`

<b style="color:#fa0240;">Handle missing entries fetched from FRED</b>

In [ ]:
fred_series_params = {
    "api_key": API_KEY,
    "file_type": "json"
}
def requester_fred_series(url="https://api.stlouisfed.org/fred/series?", base_params=fred_series_params, SeriesIDs=SeriesIDs):
    for series_id in SeriesIDs:
        params = {
            **base_params,
            "series_id": series_id
        }

        r = requests.get(url=url, params=params)

        if r.status_code == 200:
            return r.json()
        else:
            status = f"{params['series_id']} raised an error code {r.status_code}"
            return status

requester_fred_series()

{'realtime_start': '2026-08-07', 'realtime_end': '2026-08-07', 'seriess': [{'id': 'UNRATE', 'realtime_start': '2026-08-07', 'realtime_end': '2026-08-07', 'title': 'Unemployment Rate', 'observation_start': '1948-01-01', 'observation_end': '2026-07-01', 'frequency': 'Monthly', 'frequency_short': 'M', 'units': 'Percent', 'units_short': '%', 'seasonal_adjustment': 'Seasonally Adjusted', 'seasonal_adjustment_short': 'SA', 'last_updated': '2026-08-07 08:31:41-05', 'popularity': 96, 'notes': "The unemployment rate represents the number of unemployed as a percentage of the labor force. Labor force data are restricted to people 16 years of age and older, who currently reside in 1 of the 50 states or the District of Columbia, who do not reside in institutions (e.g., penal and mental facilities, homes for the aged), and who are not on active duty in the Armed Forces.\r\n\r\nThis rate is also defined as the U-3 measure of labor underutilization.\r\n\r\nThe series comes from the 'Current Population

## Get data for db table `FREDObservations`

- SeriesID
- ObservationDate
- NativeValue
- DateFetched `TEXT NOT NULL DEFAULT (datetime('now')),`

<b style="color:#fa0240;">Use for `observation_start` date the earliest common date among all data series</b>

## Prepare data from `FREDObservations` for db table `MonthlyFeatures`
- SeriesID
- MonthYear
- MonthlyValue
- DateCalculated `TEXT NOT NULL DEFAULT (datetime('now')),`

# Old

In [103]:
base_url = 'https://api.stlouisfed.org/fred/series/observations?'

base_params = {
    'api_key': API_KEY,
    'file_type': 'json',
    'sort_order': 'asc',
    "units": "lin"
}

def requester(series_id, observation_start, observation_end='9999-12-31', url=base_url):
    params = base_params.copy()
    params.update({
        'series_id': series_id,
        'observation_start': observation_start,
        'observation_end': observation_end
    })

    r = requests.get(url=url, params=params)

    if r.status_code == 200:
        return r.json()
    else:
        status = f"{params['series_id']} raised an error code {r.status_code}"
        return status

In [104]:
base_params

{'api_key': '937b94a2d841ad9bfad6ec242a4ded45',
 'file_type': 'json',
 'sort_order': 'asc',
 'units': 'lin'}

In [105]:
response = requester('ICSA', '2000-01-01')

In [118]:
response.keys()

dict_keys(['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations'])

In [110]:
df = pd.DataFrame(response['observations'])
df

,realtime_start,realtime_end,date,value
0,2026-08-04,2026-08-04,2000-01-01,286000
1,2026-08-04,2026-08-04,2000-01-08,298000
2,2026-08-04,2026-08-04,2000-01-15,289000
3,2026-08-04,2026-08-04,2000-01-22,284000
4,2026-08-04,2026-08-04,2000-01-29,285000
...,...,...,...,...
1382,2026-08-04,2026-08-04,2026-06-27,217000
1383,2026-08-04,2026-08-04,2026-07-04,217000
1384,2026-08-04,2026-08-04,2026-07-11,209000
1385,2026-08-04,2026-08-04,2026-07-18,188000


# Testing

In [57]:
payload = {
    'series_id': 'ICSA',
    'api_key': API_KEY,
    'file_type': 'json'
}

r = requests.get('https://api.stlouisfed.org/fred/series?', params=payload)

In [46]:
print(r.json())
print(f'observations_start: {r.json()['seriess'][0]['id']}')
print(f'observations_start: {r.json()['seriess'][0]['observation_start']}')
print(f'observations_end: {r.json()['seriess'][0]['observation_end']}')
print(f'frequency: {r.json()['seriess'][0]['frequency']}')
print(f'frequency_short: {r.json()['seriess'][0]['frequency_short']}')
print(f'units: {r.json()['seriess'][0]['units']}')



{'realtime_start': '2026-07-30', 'realtime_end': '2026-07-30', 'seriess': [{'id': 'ICSA', 'realtime_start': '2026-07-30', 'realtime_end': '2026-07-30', 'title': 'Initial Claims', 'observation_start': '1967-01-07', 'observation_end': '2026-07-25', 'frequency': 'Weekly, Ending Saturday', 'frequency_short': 'W', 'units': 'Number', 'units_short': 'Number', 'seasonal_adjustment': 'Seasonally Adjusted', 'seasonal_adjustment_short': 'SA', 'last_updated': '2026-07-30 07:36:19-05', 'popularity': 81, 'notes': 'An initial claim is a claim filed by an unemployed individual after a separation from an employer. The claim requests a determination of basic eligibility for the Unemployment Insurance program.'}]}
observations_start: ICSA
observations_start: 1967-01-07
observations_end: 2026-07-25
frequency: Weekly, Ending Saturday
frequency_short: W
units: Number


In [89]:
r.json()

{'realtime_start': '2026-07-30',
 'realtime_end': '2026-07-30',
 'seriess': [{'id': 'ICSA',
   'realtime_start': '2026-07-30',
   'realtime_end': '2026-07-30',
   'title': 'Initial Claims',
   'observation_start': '1967-01-07',
   'observation_end': '2026-07-25',
   'frequency': 'Weekly, Ending Saturday',
   'frequency_short': 'W',
   'units': 'Number',
   'units_short': 'Number',
   'seasonal_adjustment': 'Seasonally Adjusted',
   'seasonal_adjustment_short': 'SA',
   'last_updated': '2026-07-30 07:36:19-05',
   'popularity': 81,
   'notes': 'An initial claim is a claim filed by an unemployed individual after a separation from an employer. The claim requests a determination of basic eligibility for the Unemployment Insurance program.'}]}

In [111]:
payload.update({'frequency': 'm'})
payload.update({'sort_order': 'desc'})
r_obsv = requests.get(base_url, params=payload)

In [112]:
print(r_obsv.status_code)

200
